In [1]:
!pip install -q gdown transformers accelerate

import os, sys
from pathlib import Path
import gdown

# Dataset + helper code live in the shared IOAI-2026/AnimalDeduction/dataset folder
# (public link). Download locally and import from there — no sign-in needed.
LOCAL_DIR = Path('/content/animaldeduction')
if not LOCAL_DIR.exists() or not any(LOCAL_DIR.iterdir()):
    gdown.download_folder(id='1YheHvGfQw5YUa7MjdUF0hQC4sdtLZ5UC',
                          output=str(LOCAL_DIR), quiet=True, use_cookies=False)

sys.path.insert(0, str(LOCAL_DIR))
os.chdir(LOCAL_DIR)
print('Working directory:', os.getcwd())
print('Files:', sorted(p.name for p in LOCAL_DIR.iterdir()))

Working directory: /content/animaldeduction
Files: ['animals_pool.txt', 'dev.csv', 'evaluate.py', 'interactor.py', 'questions_pool.txt', 'test1.csv']


In [2]:
import random
import numpy as np
import pandas as pd
import torch

from interactor import Interactor
from evaluate import evaluate, load_pools

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

animals_pool, questions_pool = load_pools()
print(f'animals_pool size:   {len(animals_pool):>6}  (e.g. {animals_pool[:5]})')
print(f'questions_pool size: {len(questions_pool):>6}  (e.g. {questions_pool[:3]})')

# Sanity probe: create one Interactor and ask two questions about an octopus.
probe = Interactor(gold_animal='octopus', animals_pool=animals_pool, questions_pool=questions_pool)
print("\nask('is it a mammal?')      ->", probe.ask('is it a mammal?'))
print("ask('does it live in water?') ->", probe.ask('does it live in water?'))
print('Queries used:', probe.queries_used, '/', probe.budget)

# The model loads in bfloat16, which a T4 (Turing) cannot accelerate. Convert to
# float16 (T4 has fp16 tensor cores) -> identical answers, several times faster.
import torch
if torch.cuda.is_available() and next(Interactor._model.parameters()).dtype != torch.float16:
    Interactor._model = Interactor._model.half()
    print('oracle model -> float16 (faster on T4)')

Device: cuda
animals_pool size:     1472  (e.g. ['lion', 'tiger', 'leopard', 'snow leopard', 'cheetah'])
questions_pool size:    559  (e.g. ['is it a mammal?', 'is it a bird?', 'is it a reptile?'])
  [interactor] loading Qwen/Qwen2.5-3B-Instruct on cuda...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [interactor] LLM ready.

ask('is it a mammal?')      -> no
ask('does it live in water?') -> yes
Queries used: 2 / 15


In [3]:
class RandomBaseline:
    def __init__(self, animals_pool, questions_pool, seed=0):
        self.animals_pool = animals_pool
        self.questions_pool = questions_pool
        self.rng = random.Random(seed)

    def solve(self, interactor):
        guessed = set()
        while not interactor.is_done():
            cand = self.rng.choice(self.animals_pool)
            while cand in guessed:
                cand = self.rng.choice(self.animals_pool)
            guessed.add(cand)
            interactor.guess(cand)

baseline_results = evaluate(RandomBaseline(animals_pool, questions_pool), 'dev.csv')

  25/150 rows  mean_score=0.0000  (0.0s)
  50/150 rows  mean_score=0.0156  (0.0s)
  75/150 rows  mean_score=0.0219  (0.0s)
  100/150 rows  mean_score=0.0242  (0.1s)
  125/150 rows  mean_score=0.0194  (0.1s)
  150/150 rows  mean_score=0.0161  (0.1s)

  Dataset:       dev.csv
  Mean score:    0.0161
  Solved rate:   2.0%
  Mean queries:  14.89 / 15
  Wall time:     0.1s


In [4]:
# Reference solution (optional, slow to init). Demonstrates precompute + match,
# but uses FIXED, non-adaptive questions -> leaves most of the score on the table.
FIXED_QUESTIONS = [
    'is it a mammal?',
    'is it a bird?',
    'is it a fish?',
    'is it an insect?',
    'does it live in water?',
    'can it fly?',
    'is it a carnivore?',
    'is it bigger than a human?',
    'does it have a backbone?',
    'is it commonly kept as a pet?',
    'does it have legs?',
    'does it lay eggs?',
]

class FixedQuestionsReference:
    def __init__(self, animals_pool, questions_pool, max_animals=None):
        self.animals_pool = animals_pool
        self.questions_pool = set(questions_pool)
        self.fixed = [q for q in FIXED_QUESTIONS if q in self.questions_pool]
        from interactor import Interactor
        cand = animals_pool if max_animals is None else animals_pool[:max_animals]
        self.candidates = cand
        print(f'  [reference] precomputing {len(cand)} x {len(self.fixed)} answer table...')
        self.table = {}
        for i, a in enumerate(cand):
            sim = Interactor(gold_animal=a, animals_pool=self.animals_pool,
                             questions_pool=self.questions_pool, budget=10**9)
            self.table[a] = tuple(1 if sim.ask(q) == 'yes' else 0 for q in self.fixed)
            if (i + 1) % 200 == 0:
                print(f'    {i+1}/{len(cand)}')
        print('  [reference] table ready.')

    def solve(self, interactor):
        obs = []
        for q in self.fixed:
            if interactor.remaining_budget() <= 1:
                break
            obs.append(1 if interactor.ask(q) == 'yes' else 0)
        obs = tuple(obs)
        def agree(a):
            vec = self.table[a]
            return sum(1 for x, y in zip(vec, obs) if x == y)
        ranked = sorted(self.candidates, key=agree, reverse=True)
        for a in ranked:
            if interactor.is_done():
                break
            interactor.guess(a)

# Example (commented out by default — uncomment to run; slow to init):
# ref = FixedQuestionsReference(animals_pool, questions_pool)
# ref_results = evaluate(ref, 'dev.csv')

In [5]:
class MySolution:
    def __init__(
        self,
        animals_pool,
        questions_pool,
        number_of_questions=10,
        reserved_guesses=5,
        cache_path='fixed_baseline_cache.npz'
    ):
        self.animals_pool = list(animals_pool)
        self.questions_pool = list(questions_pool)

        self.number_of_questions = number_of_questions
        self.reserved_guesses = reserved_guesses
        self.cache_path = cache_path

        question_set = set(self.questions_pool)

        # Solo conservar preguntas que realmente aparecen en questions_pool.
        self.fixed_questions = [
            question
            for question in FIXED_QUESTIONS
            if question in question_set
        ]

        if len(self.fixed_questions) == 0:
            raise ValueError(
                'Ninguna pregunta del baseline aparece en questions_pool.'
            )

        # Cargar o construir la matriz:
        # table[animal_id][question_id] = 1 si responde yes, 0 si responde no.
        self.table = self._load_or_build_table()

        # Elegir las preguntas globalmente más balanceadas.
        self.selected_question_ids = self._select_balanced_questions()

        print('\nPreguntas seleccionadas:')

        for question_id in self.selected_question_ids:
            yes_count = int(self.table[:, question_id].sum())
            no_count = len(self.animals_pool) - yes_count

            print(
                f'  {self.fixed_questions[question_id]} '
                f'-> yes={yes_count}, no={no_count}'
            )

    def _cache_is_valid(self, cache):
        """Comprueba que el caché corresponde a estos animales y preguntas."""

        required_keys = {'animals', 'questions', 'table'}

        if not required_keys.issubset(cache.files):
            return False

        cached_animals = cache['animals']
        cached_questions = cache['questions']

        same_animals = np.array_equal(
            cached_animals,
            np.asarray(self.animals_pool)
        )

        same_questions = np.array_equal(
            cached_questions,
            np.asarray(self.fixed_questions)
        )

        expected_shape = (
            len(self.animals_pool),
            len(self.fixed_questions)
        )

        same_shape = cache['table'].shape == expected_shape

        return same_animals and same_questions and same_shape

    def _load_or_build_table(self):
        """Carga la tabla si existe; de lo contrario, la precalcula."""

        if os.path.exists(self.cache_path):
            try:
                cache = np.load(self.cache_path)

                if self._cache_is_valid(cache):
                    print(f'Cargando tabla desde {self.cache_path}')
                    return cache['table'].astype(np.uint8)

                print('El caché no corresponde a los datos actuales.')
                print('La tabla será calculada nuevamente.')

            except Exception as error:
                print('No fue posible cargar el caché:', error)

        number_of_animals = len(self.animals_pool)
        number_of_questions = len(self.fixed_questions)

        table = np.zeros(
            (number_of_animals, number_of_questions),
            dtype=np.uint8
        )

        print(
            f'Precalculando tabla de '
            f'{number_of_animals} animales x '
            f'{number_of_questions} preguntas...'
        )

        for animal_id, animal in enumerate(self.animals_pool):
            # Este Interactor usa nuestra propia copia local del modelo.
            # Sus consultas no consumen presupuesto de las partidas evaluadas.
            simulator = Interactor(
                gold_animal=animal,
                animals_pool=self.animals_pool,
                questions_pool=self.questions_pool,
                budget=10**9
            )

            for question_id, question in enumerate(self.fixed_questions):
                answer = simulator.ask(question)
                table[animal_id, question_id] = int(answer == 'yes')

            if (animal_id + 1) % 100 == 0:
                print(
                    f'  Procesados {animal_id + 1}'
                    f'/{number_of_animals} animales'
                )

        np.savez_compressed(
            self.cache_path,
            animals=np.asarray(self.animals_pool),
            questions=np.asarray(self.fixed_questions),
            table=table
        )

        print(f'Tabla guardada en {self.cache_path}')

        return table

    def _select_balanced_questions(self):
        """
        Escoge preguntas que dividen el conjunto total de animales
        lo más cerca posible de 50 % yes y 50 % no.
        """

        number_of_animals = len(self.animals_pool)

        yes_counts = self.table.sum(axis=0)

        # Distancia respecto a una división perfecta por la mitad.
        imbalance = np.abs(
            yes_counts - number_of_animals / 2
        )

        # Menor imbalance significa una pregunta más balanceada.
        ordered_question_ids = np.argsort(imbalance)

        amount = min(
            self.number_of_questions,
            len(self.fixed_questions)
        )

        return ordered_question_ids[:amount].tolist()

    def solve(self, interactor):
        observed_answers = []
        asked_question_ids = []

        # ---------------------------------------------------------
        # 1. Hacer hasta 10 preguntas fijas.
        # ---------------------------------------------------------
        for question_id in self.selected_question_ids:
            # Reservamos consultas para poder hacer guesses.
            if interactor.remaining_budget() <= self.reserved_guesses:
                break

            question = self.fixed_questions[question_id]
            answer = interactor.ask(question)

            observed_answers.append(int(answer == 'yes'))
            asked_question_ids.append(question_id)

        observed_answers = np.asarray(
            observed_answers,
            dtype=np.uint8
        )

        # ---------------------------------------------------------
        # 2. Comparar cada animal con las respuestas observadas.
        # ---------------------------------------------------------
        candidate_vectors = self.table[:, asked_question_ids]

        # Distancia de Hamming:
        # cantidad de respuestas en las que el animal no coincide.
        mismatches = np.sum(
            candidate_vectors != observed_answers,
            axis=1
        )

        # Menor cantidad de errores = mejor candidato.
        ranking = np.argsort(mismatches, kind='stable')

        # ---------------------------------------------------------
        # 3. Usar el presupuesto restante para adivinar.
        # ---------------------------------------------------------
        for animal_id in ranking:
            if interactor.is_done():
                break

            animal = self.animals_pool[animal_id]
            interactor.guess(animal)

In [6]:
import os

solution = MySolution(animals_pool, questions_pool)
dev_results   = evaluate(solution, 'dev.csv')
test1_results = evaluate(solution, 'test1.csv')

splits = [('dev', dev_results), ('test1', test1_results)]
# test2 is a held-out set; included automatically only if present in the folder.
if os.path.exists('test2.csv'):
    splits.append(('test2', evaluate(solution, 'test2.csv')))

rows = [{
    'split': name, 'n': r['n'], 'mean_score': r['mean_score'],
    'solved_rate': r['solved_rate'], 'mean_queries': r['mean_queries'],
} for name, r in splits]

# FINAL = n-weighted mean over every test split available (test1 [+ test2]).
tests = [r for name, r in splits if name.startswith('test')]
n_test = sum(r['n'] for r in tests)
rows.append({
    'split': 'FINAL',
    'n': n_test,
    'mean_score':   sum(r['mean_score']   * r['n'] for r in tests) / n_test,
    'solved_rate':  sum(r['solved_rate']  * r['n'] for r in tests) / n_test,
    'mean_queries': sum(r['mean_queries'] * r['n'] for r in tests) / n_test,
})
pd.DataFrame(rows)

Precalculando tabla de 1472 animales x 12 preguntas...
  Procesados 100/1472 animales
  Procesados 200/1472 animales
  Procesados 300/1472 animales
  Procesados 400/1472 animales
  Procesados 500/1472 animales
  Procesados 600/1472 animales
  Procesados 700/1472 animales
  Procesados 800/1472 animales
  Procesados 900/1472 animales
  Procesados 1000/1472 animales
  Procesados 1100/1472 animales
  Procesados 1200/1472 animales
  Procesados 1300/1472 animales
  Procesados 1400/1472 animales
Tabla guardada en fixed_baseline_cache.npz

Preguntas seleccionadas:
  does it lay eggs? -> yes=760, no=712
  does it have legs? -> yes=847, no=625
  does it live in water? -> yes=532, no=940
  is it a mammal? -> yes=433, no=1039
  does it have a backbone? -> yes=1048, no=424
  is it a carnivore? -> yes=423, no=1049
  can it fly? -> yes=331, no=1141
  is it a bird? -> yes=329, no=1143
  is it a fish? -> yes=229, no=1243
  is it bigger than a human? -> yes=210, no=1262
  25/150 rows  mean_score=0.1472 

,split,n,mean_score,solved_rate,mean_queries
0,dev,150,0.221467,0.293333,14.193333
1,test1,500,0.240920,0.322000,14.224000
2,FINAL,500,0.240920,0.322000,14.224000
